# EDA

This notebook is the shared workspace for Stage 4 sample EDA and the Stage 4a patch flow.


In [1]:
%reset -f
# Resolve the project root no matter whether Jupyter starts in the repo root or the notebook directory.
from pathlib import Path
import datetime as dt
import sys

candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve() / "steam-crawler",
    Path.cwd().resolve().parent / "steam-crawler",
]
ROOT_DIR = next(path for path in candidates if (path / "requirements.txt").exists())
REQUIREMENTS_FILE = ROOT_DIR / "requirements.txt"
ROOT_DIR


PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler')

In [2]:
# Install every runtime dependency declared in requirements.txt into the current notebook kernel environment.
print(f"Installing notebook dependencies from {REQUIREMENTS_FILE}")
!{sys.executable} -m pip install -r "{REQUIREMENTS_FILE}"

SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

SRC_DIR


Installing notebook dependencies from /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/requirements.txt


PosixPath('/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/src')

In [3]:
# Load the crawler env with overrides enabled, resolve the configured data directory, and validate the Kaggle credentials.
import importlib
import os
import shutil
from pathlib import Path
from tempfile import TemporaryDirectory

import kagglehub
import steam_crawler.config as steam_config

steam_config = importlib.reload(steam_config)

ENV_PATH = steam_config.load_project_env(ROOT_DIR, override=True)
DATA_DIR = steam_config.resolve_data_dir(ROOT_DIR)
STAGE_04_PATH = DATA_DIR / "stage_04_selected_games.csv"
STAGE_04A_CSV_PATH = DATA_DIR / "stage_04a_selected_games.csv"
STAGE_04A_PARQUET_PATH = DATA_DIR / "raw_selected_games.parquet"
STAGE_05A_PARQUET_PATH = DATA_DIR / "raw_reviews_dataset.parquet"
KAGGLE_SHARED_DATASET_HANDLE = f"{os.getenv('KAGGLE_USERNAME', '')}/steam-cs5242-dataset" if os.getenv('KAGGLE_USERNAME') else None

required_env = {
    "KAGGLE_USERNAME": os.getenv("KAGGLE_USERNAME"),
    "KAGGLE_API_TOKEN": os.getenv("KAGGLE_API_TOKEN"),
}
missing_env = [name for name, value in required_env.items() if not value]
if missing_env:
    raise RuntimeError(
        "Missing required Kaggle credentials: "
        + ", ".join(missing_env)
        + ". Add them to the environment or to steam-crawler/.env before running this notebook."
    )

KAGGLE_USERNAME = required_env["KAGGLE_USERNAME"]
KAGGLE_SHARED_DATASET_HANDLE = f"{KAGGLE_USERNAME}/steam-cs5242-dataset"
os.environ.setdefault("KAGGLE_KEY", required_env["KAGGLE_API_TOKEN"])

def upload_kaggle_dataset_snapshot(
    dataset_handle: str,
    dataset_files: dict[str, Path],
    *,
    version_notes: str = "",
) -> dict[str, object]:
    if dataset_handle.count("/") != 1:
        raise ValueError(
            "Kaggle dataset handle must look like '<KAGGLE_USERNAME>/<DATASET_SLUG>'."
        )

    staged_files: dict[str, str] = {}
    with TemporaryDirectory(prefix="kagglehub_dataset_") as tmp_dir_name:
        tmp_dir = Path(tmp_dir_name)
        for target_name, source_path in dataset_files.items():
            source_path = Path(source_path)
            if not source_path.exists():
                continue
            staged_path = tmp_dir / target_name
            staged_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, staged_path)
            staged_files[target_name] = str(source_path)

        if not staged_files:
            raise FileNotFoundError(
                "No source parquet files were found for the Kaggle dataset snapshot."
            )

        kagglehub.dataset_upload(
            dataset_handle,
            str(tmp_dir),
            version_notes=version_notes,
        )

    dataset_url = f"https://www.kaggle.com/datasets/{dataset_handle}"
    print(f"Uploaded {len(staged_files)} parquet file(s) to {dataset_url}")
    for target_name, source_path in staged_files.items():
        print(f"- {target_name}: {source_path}")
    return {
        "handle": dataset_handle,
        "url": dataset_url,
        "files": staged_files,
        "version_notes": version_notes,
    }

if not STAGE_04_PATH.exists():
    raise FileNotFoundError(f"Stage 4 output not found: {STAGE_04_PATH}")

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"ENV_PATH: {ENV_PATH}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"STAGE_04_PATH: {STAGE_04_PATH}")
print(f"STAGE_04A_CSV_PATH: {STAGE_04A_CSV_PATH}")
print(f"STAGE_04A_PARQUET_PATH: {STAGE_04A_PARQUET_PATH}")
print(f"STAGE_05A_PARQUET_PATH: {STAGE_05A_PARQUET_PATH}")
print(f"Kaggle credentials: ready for kagglehub as {KAGGLE_USERNAME}")
print(f"Shared Kaggle dataset handle: {KAGGLE_SHARED_DATASET_HANDLE}")


ROOT_DIR: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler
ENV_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/.env
DATA_DIR: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data
STAGE_04_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04_selected_games.csv
STAGE_04A_CSV_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04a_selected_games.csv
STAGE_04A_PARQUET_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet
STAGE_05A_PARQUET_PATH: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_reviews_dataset.parquet
Kaggle credentials: ready for kagglehub as gitaalekhyapaul
Shared Kaggle dataset handle: gitaalekhyapaul/steam-cs5242-dataset


In [4]:
import pandas as pd
from IPython.display import display

stage_04_df = pd.read_csv(STAGE_04_PATH)
print(f"Loaded Stage 4 dataframe: {stage_04_df.shape[0]:,} rows x {stage_04_df.shape[1]} columns")
display(stage_04_df.head())


Loaded Stage 4 dataframe: 9,598 rows x 15 columns


,appid,name,last_modified,price_change_number,raw_app_json,details_success,type,category_ids,category_descriptions,recommendations_total,raw_details_json,eligible_for_sampling,sample_rank,random_seed,sampled_at
0,10,Counter-Strike,1745368572,34672985,"{""appid"":10,""name"":""Counter-Strike"",""last_modi...",True,game,1|49|36|37|66|68|75|69|8|62,Multi-player|PvP|Online PvP|Shared/Split Scree...,167351,"{""10"":{""success"":true,""data"":{""type"":""game"",""n...",True,1,5242,2026-04-15T15:59:56.102758+00:00
1,20,Team Fortress Classic,1745368565,34672985,"{""appid"":20,""name"":""Team Fortress Classic"",""la...",True,game,1|49|36|37|68|75|69|8|44|62,Multi-player|PvP|Online PvP|Shared/Split Scree...,6883,"{""20"":{""success"":true,""data"":{""type"":""game"",""n...",True,2,5242,2026-04-15T15:59:56.102758+00:00
2,30,Day of Defeat,1745368580,34672985,"{""appid"":30,""name"":""Day of Defeat"",""last_modif...",True,game,1|67|66|68|69|8|62,Multi-player|Camera Comfort|Color Alternatives...,4419,"{""30"":{""success"":true,""data"":{""type"":""game"",""n...",True,3,5242,2026-04-15T15:59:56.102758+00:00
3,40,Deathmatch Classic,1745368570,34672985,"{""appid"":40,""name"":""Deathmatch Classic"",""last_...",True,game,1|49|36|37|66|68|75|69|8|44|62,Multi-player|PvP|Online PvP|Shared/Split Scree...,2403,"{""40"":{""success"":true,""data"":{""type"":""game"",""n...",True,4,5242,2026-04-15T15:59:56.102758+00:00
4,50,Half-Life: Opposing Force,1745368539,34672985,"{""appid"":50,""name"":""Half-Life: Opposing Force""...",True,game,2|1|68|78|74|79|8|62,Single-player|Multi-player|Custom Volume Contr...,24416,"{""50"":{""success"":true,""data"":{""type"":""game"",""n...",True,5,5242,2026-04-15T15:59:56.102758+00:00


In [5]:
print(f"Unique sampled appids: {stage_04_df['appid'].nunique():,}")
display(stage_04_df.isna().sum().rename('null_count').to_frame())

for label in ['details_success', 'type', 'eligible_for_sampling']:
    if label not in stage_04_df.columns:
        continue
    print(f"{label}")
    display(
        stage_04_df[label]
        .value_counts(dropna=False)
        .rename_axis('value')
        .reset_index(name='count')
    )

if 'recommendations_total' in stage_04_df.columns:
    recommendations_series = pd.to_numeric(stage_04_df['recommendations_total'], errors='coerce')
    display(recommendations_series.describe().to_frame(name='recommendations_total'))


Unique sampled appids: 9,598


,null_count
appid,0
name,0
last_modified,0
price_change_number,0
raw_app_json,0
details_success,0
type,0
category_ids,1
category_descriptions,1
recommendations_total,0


details_success


,value,count
0,True,9598


type


,value,count
0,game,9598


eligible_for_sampling


,value,count
0,True,9598


,recommendations_total
count,9.598000e+03
mean,1.113040e+04
std,7.221442e+04
min,5.010000e+02
25%,8.310000e+02
50%,1.675000e+03
75%,4.715250e+03
max,5.029425e+06


## Stage 4a Patch

This patch uses the sampled Stage 4 CSV as the source of app ids, recommendations totals, and category ids.
It re-fetches only the missing store metadata needed for price and one review page per game to capture the review summary.


In [6]:
import steam_crawler.stage4a as steam_stage4a

STAGE_04A_ENDPOINT_MODE = os.getenv("STAGE_04A_ENDPOINT_MODE") or None
STAGE_04A_FORCE_REFRESH = False

stage_04a_csv_df = steam_stage4a.build_stage_04a(
    ROOT_DIR,
    force_refresh=STAGE_04A_FORCE_REFRESH,
    endpoint_mode=STAGE_04A_ENDPOINT_MODE,
)

display(stage_04a_csv_df.head())
display(stage_04a_csv_df.describe(include="all").transpose())
print(f"stage_04a csv: {STAGE_04A_CSV_PATH}")


Stage 4a patch: 100%|##########| 9598/9598 [00:00<?, ?apps/s]

2026-04-22 14:27:36,636 | INFO | stage_04a csv summary | rows=9598 | csv=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04a_selected_games.csv | retries=0 | errors=0


,id,num_reviews,%positive_reviews,price,app_category
0,10,167351,97.417145,9.99,1|49|36|37|66|68|75|69|8|62
1,20,6883,87.119798,4.99,1|49|36|37|68|75|69|8|44|62
2,30,4419,90.234536,4.99,1|67|66|68|69|8|62
3,40,2403,83.216142,4.99,1|49|36|37|66|68|75|69|8|44|62
4,50,24416,95.340014,4.99,2|1|68|78|74|79|8|62


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,9598.0,<NA>,<NA>,<NA>,1247122.349344,950579.33743,10.0,431195.0,1062820.0,1843820.0,4450800.0
num_reviews,9598.0,<NA>,<NA>,<NA>,11130.399771,72214.416315,501.0,831.0,1675.0,4715.25,5029425.0
%positive_reviews,9598.0,<NA>,<NA>,<NA>,83.606166,11.923904,9.991742,77.785483,86.523173,92.515801,99.717442
price,9525.0,<NA>,<NA>,<NA>,15.834571,13.026388,0.0,5.99,12.99,19.99,99.99
app_category,9597,3433,2|22|28|23|62,498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


stage_04a csv: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_04a_selected_games.csv


In [7]:
stage_04a_df = steam_stage4a.write_stage_04a_parquet(
    ROOT_DIR,
    endpoint_mode=STAGE_04A_ENDPOINT_MODE,
)

display(stage_04a_df.head())
display(stage_04a_df.describe(include="all").transpose())
print(f"stage_04a parquet: {STAGE_04A_PARQUET_PATH}")


,id,num_reviews,%positive_reviews,price,app_category
0,10,167351,97.417145,9.99,1|49|36|37|66|68|75|69|8|62
1,20,6883,87.119798,4.99,1|49|36|37|68|75|69|8|44|62
2,30,4419,90.234536,4.99,1|67|66|68|69|8|62
3,40,2403,83.216142,4.99,1|49|36|37|66|68|75|69|8|44|62
4,50,24416,95.340014,4.99,2|1|68|78|74|79|8|62


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,9598.0,<NA>,<NA>,<NA>,1247122.349344,950579.33743,10.0,431195.0,1062820.0,1843820.0,4450800.0
num_reviews,9598.0,<NA>,<NA>,<NA>,11130.399771,72214.416315,501.0,831.0,1675.0,4715.25,5029425.0
%positive_reviews,9598.0,<NA>,<NA>,<NA>,83.606166,11.923904,9.991742,77.785483,86.523173,92.515801,99.717442
price,9525.0,<NA>,<NA>,<NA>,15.834571,13.026388,0.0,5.99,12.99,19.99,99.99
app_category,9597,3433,2|22|28|23|62,498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


stage_04a parquet: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet


In [8]:
# Edit the shared dataset handle here if you want a different Kaggle dataset slug.
KAGGLE_SHARED_DATASET_HANDLE = f"{KAGGLE_USERNAME}/steam-cs5242-dataset"

# Each parquet file becomes a separate tabular resource inside the same Kaggle dataset.
stage_04a_kaggle_dataset_files = {
    "raw_selected_games.parquet": STAGE_04A_PARQUET_PATH,
    "raw_reviews_dataset.parquet": STAGE_05A_PARQUET_PATH,
}

stage_04a_kaggle_upload = upload_kaggle_dataset_snapshot(
    KAGGLE_SHARED_DATASET_HANDLE,
    stage_04a_kaggle_dataset_files,
    version_notes=f"Refresh shared stage parquet dataset after stage_04a_selected_games update {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
)

stage_04a_kaggle_upload


Uploading Dataset https://api.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset ...
Starting upload for file /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_dataset_pey4c7yk/raw_selected_games.parquet


Uploading: 100%|██████████| 258k/258k [00:01<00:00, 171kB/s]

Upload successful: /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_dataset_pey4c7yk/raw_selected_games.parquet (252KB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset
Uploaded 1 parquet file(s) to https://www.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset
- raw_selected_games.parquet: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet


{'handle': 'gitaalekhyapaul/steam-cs5242-dataset',
 'url': 'https://www.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset',
 'files': {'raw_selected_games.parquet': '/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet'},
 'version_notes': 'Refresh shared stage parquet dataset after stage_04a_selected_games update 2026-04-22 14:27:36'}

## Stage 5 EDA

This section inspects the staged review dataset before the Stage 5a transform.
It previews rows and summarizes a few high-signal fields from the stored review payloads.


In [9]:
import csv
import gzip
import json
from collections import Counter

STAGE_05_PATH = DATA_DIR / "stage_05_reviews_dataset.csv.gz"

STAGE_05_CSV_FIELD_SIZE_LIMIT_READY = False


def configure_stage_05_csv_field_size_limit() -> None:
    global STAGE_05_CSV_FIELD_SIZE_LIMIT_READY
    if STAGE_05_CSV_FIELD_SIZE_LIMIT_READY:
        return
    limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limit)
            STAGE_05_CSV_FIELD_SIZE_LIMIT_READY = True
            return
        except OverflowError:
            limit //= 10


def open_stage_05_text(path: Path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8", newline="")
    return path.open("rt", encoding="utf-8", newline="")


def stage_05_fieldnames(path: Path) -> list[str]:
    configure_stage_05_csv_field_size_limit()
    with open_stage_05_text(path) as handle:
        reader = csv.DictReader(handle)
        return reader.fieldnames or []


def preview_stage_05_rows(path: Path, limit: int = 5) -> pd.DataFrame:
    configure_stage_05_csv_field_size_limit()
    with open_stage_05_text(path) as handle:
        return pd.read_csv(handle, nrows=limit)


def parse_stage_05_raw_json(raw_json: str) -> dict:
    if not isinstance(raw_json, str) or not raw_json:
        return {}
    try:
        payload = json.loads(raw_json)
    except json.JSONDecodeError:
        return {}
    return payload if isinstance(payload, dict) else {}


def summarize_stage_05(path: Path) -> tuple[int, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    configure_stage_05_csv_field_size_limit()
    row_count = 0
    source_counter: Counter[str] = Counter()
    voted_up_counter: Counter[str] = Counter()
    language_counter: Counter[str] = Counter()

    with open_stage_05_text(path) as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row_count += 1
            source_counter[str(row.get('source_stream', ''))] += 1
            payload = parse_stage_05_raw_json(row.get('raw_json', ''))
            voted_up_counter[str(payload.get('voted_up', ''))] += 1
            language_counter[str(payload.get('language', ''))] += 1

    source_df = pd.DataFrame(
        sorted(source_counter.items(), key=lambda item: (-item[1], item[0])),
        columns=['source_stream', 'count'],
    )
    voted_up_df = pd.DataFrame(
        sorted(voted_up_counter.items(), key=lambda item: (-item[1], item[0])),
        columns=['voted_up', 'count'],
    )
    language_df = pd.DataFrame(
        sorted(language_counter.items(), key=lambda item: (-item[1], item[0])),
        columns=['language', 'count'],
    )
    return row_count, source_df, voted_up_df, language_df


In [10]:
if not STAGE_05_PATH.exists():
    raise FileNotFoundError(f"Stage 5 output not found: {STAGE_05_PATH}")

stage_05_fieldnames_list = stage_05_fieldnames(STAGE_05_PATH)
stage_05_preview_df = preview_stage_05_rows(STAGE_05_PATH)
stage_05_row_count, stage_05_source_df, stage_05_voted_up_df, stage_05_language_df = summarize_stage_05(STAGE_05_PATH)

print(f"stage_05 source: {STAGE_05_PATH}")
print(f"stage_05 rows: {stage_05_row_count:,}")
print(f"stage_05 columns ({len(stage_05_fieldnames_list)}): {stage_05_fieldnames_list}")

display(stage_05_preview_df)
display(stage_05_source_df)
display(stage_05_voted_up_df)
display(stage_05_language_df.head(20))


stage_05 source: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_reviews_dataset.csv.gz
stage_05 rows: 4,319,210
stage_05 columns (7): ['appid', 'recommendationid', 'author_steamid', 'timestamp_created', 'review_text', 'source_stream', 'raw_json']


,appid,recommendationid,author_steamid,timestamp_created,review_text,source_stream,raw_json
0,10,223297816,76561198018492187,1776246951,обосрался от катки,recent,"{""recommendationid"":""223297816"",""author"":{""ste..."
1,10,223296465,76561199524057416,1776244567,Spoko gra,recent,"{""recommendationid"":""223296465"",""author"":{""ste..."
2,10,223295108,76561198958671560,1776242134,gfugjghj,recent,"{""recommendationid"":""223295108"",""author"":{""ste..."
3,10,223294997,76561199031337761,1776241948,Q,recent,"{""recommendationid"":""223294997"",""author"":{""ste..."
4,10,223294897,76561199751459206,1776241794,пиратка лучше,recent,"{""recommendationid"":""223294897"",""author"":{""ste..."


,source_stream,count
0,recent,4318810
1,helpful,400


,voted_up,count
0,True,3509838
1,False,809372


,language,count
0,english,2049292
1,schinese,646357
2,russian,471973
3,brazilian,180671
4,spanish,165408
5,german,135297
6,french,102830
7,koreana,86466
8,turkish,86241
9,polish,77058


## Stage 5a Transform

This transform derives a narrow review dataset from the Stage 5 review rows.
It keeps only the review timestamp, Steam user id, app id, binary review score, and review upvote count.


In [11]:
import steam_crawler.stage5a as steam_stage5a

STAGE_05_PATH = DATA_DIR / "stage_05_reviews_dataset.csv.gz"
STAGE_05A_CSV_GZ_PATH = DATA_DIR / "stage_05a_reviews_dataset.csv.gz"
STAGE_05A_PARQUET_PATH = DATA_DIR / "raw_reviews_dataset.parquet"
STAGE_05A_FORCE_REFRESH = False

stage_05a_csv_summary = steam_stage5a.build_stage_05a_csv(
    ROOT_DIR,
    force_refresh=STAGE_05A_FORCE_REFRESH,
)
stage_05a_preview_df = steam_stage5a.preview_stage_05a(ROOT_DIR)

print(f"stage_05 source: {STAGE_05_PATH}")
print(f"stage_05a csv.gz: {stage_05a_csv_summary['path']}")
print(f"stage_05a rows: {stage_05a_csv_summary['rows']:,}")
print(f"stage_05a reused existing csv: {stage_05a_csv_summary['skipped']}")
display(stage_05a_preview_df)


Stage 5a csv: 0reviews [00:00, ?reviews/s]

stage_05 source: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05_reviews_dataset.csv.gz
stage_05a csv.gz: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/stage_05a_reviews_dataset.csv.gz
stage_05a rows: 4,319,210
stage_05a reused existing csv: False


,timestamp,user_id,app_id,review_id,review_score,review_rating
0,1776246951,76561198018492187,10,223297816,1,0
1,1776244567,76561199524057416,10,223296465,1,0
2,1776242134,76561198958671560,10,223295108,1,0
3,1776241948,76561199031337761,10,223294997,1,0
4,1776241794,76561199751459206,10,223294897,-1,0


In [12]:
stage_05a_parquet_summary = steam_stage5a.write_stage_05a_parquet(ROOT_DIR)
stage_05a_preview_df = steam_stage5a.preview_stage_05a(ROOT_DIR)

print(f"stage_05a parquet: {stage_05a_parquet_summary['path']}")
print(f"stage_05a parquet rows: {stage_05a_parquet_summary['rows']:,}")
print(f"stage_05a reused existing parquet: {stage_05a_parquet_summary['skipped']}")
display(stage_05a_preview_df)


Stage 5a parquet: 0rows [00:00, ?rows/s]

stage_05a parquet: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_reviews_dataset.parquet
stage_05a parquet rows: 4,319,210
stage_05a reused existing parquet: False


,timestamp,user_id,app_id,review_id,review_score,review_rating
0,1776246951,76561198018492187,10,223297816,1,0
1,1776244567,76561199524057416,10,223296465,1,0
2,1776242134,76561198958671560,10,223295108,1,0
3,1776241948,76561199031337761,10,223294997,1,0
4,1776241794,76561199751459206,10,223294897,-1,0


In [13]:
# Reuse the same Kaggle dataset handle so both stage parquet files live in one dataset.
KAGGLE_SHARED_DATASET_HANDLE = f"{KAGGLE_USERNAME}/steam-cs5242-dataset"

stage_05a_kaggle_dataset_files = {
    "raw_selected_games.parquet": STAGE_04A_PARQUET_PATH,
    "raw_reviews_dataset.parquet": STAGE_05A_PARQUET_PATH,
}

stage_05a_kaggle_upload = upload_kaggle_dataset_snapshot(
    KAGGLE_SHARED_DATASET_HANDLE,
    stage_05a_kaggle_dataset_files,
    version_notes=f"Refresh shared stage parquet dataset after stage_05a_reviews_dataset update {dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
)

stage_05a_kaggle_upload


Uploading Dataset https://api.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset ...
Starting upload for file /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_dataset_2q2aiaw_/raw_selected_games.parquet


Uploading: 100%|██████████| 258k/258k [00:01<00:00, 167kB/s]

Upload successful: /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_dataset_2q2aiaw_/raw_selected_games.parquet (252KB)
Starting upload for file /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_dataset_2q2aiaw_/raw_reviews_dataset.parquet



Uploading: 100%|██████████| 103M/103M [00:12<00:00, 8.41MB/s] 

Upload successful: /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_dataset_2q2aiaw_/raw_reviews_dataset.parquet (99MB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset
Uploaded 2 parquet file(s) to https://www.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset
- raw_selected_games.parquet: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet
- raw_reviews_dataset.parquet: /Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_reviews_dataset.parquet


{'handle': 'gitaalekhyapaul/steam-cs5242-dataset',
 'url': 'https://www.kaggle.com/datasets/gitaalekhyapaul/steam-cs5242-dataset',
 'files': {'raw_selected_games.parquet': '/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet',
  'raw_reviews_dataset.parquet': '/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_reviews_dataset.parquet'},
 'version_notes': 'Refresh shared stage parquet dataset after stage_05a_reviews_dataset update 2026-04-22 14:31:06'}

## Kaggle Sanity Check

This section downloads the shared Kaggle dataset back to a temporary directory,
confirms that both parquet resources are present, and previews the downloaded files.


In [15]:
import time

import pandas as pd
from IPython.display import display

KAGGLE_SHARED_DATASET_HANDLE = f"{KAGGLE_USERNAME}/steam-cs5242-dataset"
KAGGLE_EXPECTED_DATASET_FILES = {
    "raw_selected_games.parquet": STAGE_04A_PARQUET_PATH,
    "raw_reviews_dataset.parquet": STAGE_05A_PARQUET_PATH,
}
KAGGLE_DOWNLOAD_MAX_ATTEMPTS = 6
KAGGLE_DOWNLOAD_RETRY_SECONDS = 10

with TemporaryDirectory(prefix="kagglehub_verify_") as tmp_dir_name:
    base_download_dir = Path(tmp_dir_name)
    downloaded_paths: dict[str, Path] = {}
    missing_files: list[str] = []

    for attempt in range(1, KAGGLE_DOWNLOAD_MAX_ATTEMPTS + 1):
        attempt_dir = base_download_dir / f"attempt_{attempt}"
        attempt_dir.mkdir(parents=True, exist_ok=True)
        resolved_download_dir = Path(
            kagglehub.dataset_download(
                KAGGLE_SHARED_DATASET_HANDLE,
                output_dir=str(attempt_dir),
                force_download=True,
            )
        )
        downloaded_paths = {
            resource_name: resolved_download_dir / resource_name
            for resource_name in KAGGLE_EXPECTED_DATASET_FILES
        }
        missing_files = [
            resource_name
            for resource_name, downloaded_path in downloaded_paths.items()
            if not downloaded_path.exists()
        ]
        if not missing_files:
            print(f"Downloaded Kaggle dataset snapshot from {resolved_download_dir}")
            break
        if attempt == KAGGLE_DOWNLOAD_MAX_ATTEMPTS:
            raise RuntimeError(
                "Missing Kaggle parquet resources after "
                f"{KAGGLE_DOWNLOAD_MAX_ATTEMPTS} attempts: {missing_files}"
            )
        print(
            f"Attempt {attempt}/{KAGGLE_DOWNLOAD_MAX_ATTEMPTS}: still missing {missing_files}. "
            f"Waiting {KAGGLE_DOWNLOAD_RETRY_SECONDS} seconds for Kaggle dataset processing before retrying."
        )
        time.sleep(KAGGLE_DOWNLOAD_RETRY_SECONDS)

    for resource_name, local_source_path in KAGGLE_EXPECTED_DATASET_FILES.items():
        downloaded_path = downloaded_paths[resource_name]
        if not downloaded_path.exists():
            raise FileNotFoundError(
                f"Expected Kaggle resource missing after download: {downloaded_path}"
            )
        downloaded_df = pd.read_parquet(downloaded_path)
        print(
            f"{resource_name}: uploaded successfully | local source={local_source_path} | "
            f"downloaded rows={downloaded_df.shape[0]:,} | columns={downloaded_df.shape[1]}"
        )
        print(f"Downloaded path: {downloaded_path}")
        print("Head")
        display(downloaded_df.head())
        print("Tail")
        display(downloaded_df.tail())


100%|██████████| 82.6M/82.6M [00:05<00:00, 15.3MB/s]

Extracting files...


Downloaded Kaggle dataset snapshot from /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_verify_yci76d3w/attempt_1
raw_selected_games.parquet: uploaded successfully | local source=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_selected_games.parquet | downloaded rows=9,598 | columns=5
Downloaded path: /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_verify_yci76d3w/attempt_1/raw_selected_games.parquet
Head


,id,num_reviews,%positive_reviews,price,app_category
0,10,167351,97.417145,9.99,1|49|36|37|66|68|75|69|8|62
1,20,6883,87.119798,4.99,1|49|36|37|68|75|69|8|44|62
2,30,4419,90.234536,4.99,1|67|66|68|69|8|62
3,40,2403,83.216142,4.99,1|49|36|37|66|68|75|69|8|44|62
4,50,24416,95.340014,4.99,2|1|68|78|74|79|8|62


Tail


,id,num_reviews,%positive_reviews,price,app_category
9593,4286550,964,80.754352,6.99,2|22|23|62
9594,4298880,687,83.971631,4.99,2|22|67|68|74|23|62
9595,4309030,739,99.247177,6.99,2|22|74|79|69|23|62
9596,4327530,790,97.766749,3.99,2|22|74|23|62
9597,4450800,504,97.81719,2.99,2|22|28|74|69|70|62


raw_reviews_dataset.parquet: uploaded successfully | local source=/Users/gitaalekhyapaul/Documents/[Local] CS5242/cs5242-project/steam-crawler/data/raw_reviews_dataset.parquet | downloaded rows=4,319,210 | columns=6
Downloaded path: /var/folders/fg/p6f6x7z11qb2jtk9d1gdsdlw0000gn/T/kagglehub_verify_yci76d3w/attempt_1/raw_reviews_dataset.parquet
Head


,timestamp,user_id,app_id,review_id,review_score,review_rating
0,1776246951,76561198018492187,10,223297816,1,0
1,1776244567,76561199524057416,10,223296465,1,0
2,1776242134,76561198958671560,10,223295108,1,0
3,1776241948,76561199031337761,10,223294997,1,0
4,1776241794,76561199751459206,10,223294897,-1,0


Tail


,timestamp,user_id,app_id,review_id,review_score,review_rating
4319205,1758189558,76561199111143479,2689470,204608606,1,0
4319206,1758170523,76561199442964166,2689470,204598011,1,0
4319207,1758106651,76561199120569580,2689470,204542860,1,0
4319208,1758079721,76561198081439726,2689470,204526031,1,0
4319209,1758076836,76561199089148979,2689470,204523936,1,0
